# Decorators – Practice Exercises

This notebook contains hands-on exercises to practice **Python decorators**,
following the concepts from `decorators.ipynb`.

For each exercise:

- Read the description carefully.
- Implement your solution in the provided code cell (in the `# TODO` block).
- You can add additional helper functions/classes if needed.
- Try to run simple tests in the same cell or in new cells below.

### Contents

- [Exercise 1 – Basic Logging Decorator](#exercise-1--basic-logging-decorator)
- [Exercise 2 – Retry Decorator with Arguments](#exercise-2--retry-decorator-with-arguments)
- [Exercise 3 – Preserving Metadata with functools.wraps](#exercise-3--preserving-metadata-with-functoolswraps)
- [Exercise 4 – REST Error-Handling Decorator](#exercise-4--rest-error-handling-decorator)
- [Exercise 5 – Async WebSocket Handler Decorator](#exercise-5--async-websocket-handler-decorator)
- [Exercise 6 – Stacking Decorators and Call Order](#exercise-6--stacking-decorators-and-call-order)
- [Exercise 7 – Simple Caching (Memoization) Decorator](#exercise-7--simple-caching-memoization-decorator)
- [Exercise 8 – Caching Decorator (Class-Based)](#exercise-8--caching-decorator-class-based)

## Exercise 1 – Basic Logging Decorator

**Goal**: Implement a simple decorator `log_calls` that logs function name,
arguments, and return value.

Requirements:

- `log_calls` should work on any function with any arguments.
- When the wrapped function is called, print a line like:
  - `Calling foo with args=(1, 2), kwargs={'x': 3}`
- After the function returns, print:
  - `foo returned 42`
- Then return the original result.

Hints:

- Use `*args` and `**kwargs` in the wrapper.
- Remember to **return** the result of the wrapped function.

You can test with a simple function like `add(a, b)`.

In [4]:
# Exercise 1 – implement log_calls here

from collections.abc import Callable
from typing import Any


def log_calls(func: Callable[..., Any]) -> Callable[..., Any]:
    """Decorator that logs function name, args, kwargs, and return value.

    TODO: replace 'pass' with your implementation.
    """

    def wrapper(*args: Any, **kwargs: Any) -> Any:
        # TODO: print before-call message
        # TODO: call the original function and capture the result
        # TODO: print after-call message
        # TODO: return the result
        result = func(*args, **kwargs)
        entry = f">> {func.__name__}({args}; {kwargs}) = {result}"
        print(entry)
        return result

    return wrapper


# Example usage (uncomment to test after implementing):
@log_calls
def add(a, b):
    return a + b

print(add(1, 2))

>> add((1, 2); {}) = 3
3


## Exercise 2 – Retry Decorator with Arguments

**Goal**: Implement a **configurable** retry decorator `retry` that retries a
function when it raises an exception.

Requirements:

- Usage:
  - `@retry(max_attempts=3, delay=0.1)` above a function.
- Behavior:
  - Call the wrapped function.
  - If it raises an exception, wait `delay` seconds and try again.
  - Stop after `max_attempts` attempts; if still failing, re-raise the last exception.
- Should work for any function signature.

Hints:

- `retry` is a **decorator factory**: it returns the actual decorator.
- Inside the wrapper, use a loop and `time.sleep(delay)`.
- For this exercise, you don’t need to preserve metadata (that comes next).

You can test with a function that fails a couple of times before succeeding.

In [11]:
# Exercise 2 – implement retry(max_attempts, delay) here

import time
from collections.abc import Callable
from typing import Any


def retry(max_attempts: int = 3, delay: float = 0.0) -> Callable[[Callable[..., Any]], Callable[..., Any]]:
    """Retry decorator factory.

    On exception, retries the wrapped function up to max_attempts times,
    waiting 'delay' seconds between attempts.
    """

    def decorator(func: Callable[..., Any]) -> Callable[..., Any]:
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            """Decorator's wrapper"""
            # TODO: implement retry loop
            # - try calling func
            # - on exception, sleep and retry
            # - after max_attempts, re-raise
            for retry in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as EXC:
                    retries = max_attempts - retry
                    print(f"Error executing \"{func.__name__}({args}; {kwargs})\"")
                    print(f"Exception: \"{repr(EXC)}\"\nRetries left: {retries - 1}")
                    time.sleep(delay)

            raise Exception(f"Max attempts ({max_attempts}) reached without success")

        return wrapper

    return decorator


# Example (uncomment to test):
failures = {"count": 0}

@retry(max_attempts=3, delay=0.1)
def flaky():
    if failures["count"] < 2:
       failures["count"] += 1
       raise RuntimeError("temporary error")
    return "ok"

print(f">> Function's name: \"{flaky.__name__}\"")
print(f">> Function's description: \"{flaky.__doc__}\"")
print(">> Function's response...")
print(flaky())

>> Function's name: "wrapper"
>> Function's description: "Decorator's wrapper"
>> Function's response...
Error executing "flaky((); {})"
Exception: "RuntimeError('temporary error')"
Retries left: 2
Error executing "flaky((); {})"
Exception: "RuntimeError('temporary error')"
Retries left: 1
ok


## Exercise 3 – Preserving Metadata with `functools.wraps`

**Goal**: Modify your `retry` decorator (or write `retry_with_wraps`) so that the
wrapped function **preserves the original function’s metadata**:

- `__name__`
- `__doc__`
- `__module__`, etc.

Requirements:

- Use `functools.wraps(func)` on your inner wrapper.
- Demonstrate that `wrapped.__name__ == func.__name__` and `wrapped.__doc__ == func.__doc__`.

Hints:

- The pattern is:

  ```python
  from functools import wraps

  def decorator(func):
      @wraps(func)
      def wrapper(...):
          ...
      return wrapper
  ```

You can adapt Exercise 2’s `retry` decorator for this.

In [29]:
# Exercise 3 – implement retry_with_wraps here

from functools import wraps
from collections.abc import Callable
from typing import Any
import time

def retry_with_wraps(max_attempts: int = 3, delay: float = 0.0) -> Callable[[Callable[..., Any]], Callable[..., Any]]:
    """Retry decorator that preserves function metadata using functools.wraps.

    TODO: base the behavior on Exercise 2, but add @wraps.
    """

    def decorator(func: Callable[..., Any]) -> Callable[..., Any]:
        @wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            # TODO: implement retry logic (similar to Exercise 2)
            for retry in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as EXC:
                    retries = max_attempts - retry
                    print(f"Error executing \"{func.__name__}({args}; {kwargs})\"")
                    print(f"Exception: \"{repr(EXC)}\"\nRetries left: {retries - 1}")
                    time.sleep(delay)

            raise Exception(f"Max attempts ({max_attempts}) reached without success")

        return wrapper

    return decorator


# Example (uncomment to test after implementation):
@retry_with_wraps(max_attempts=2)
def compute(x: int) -> int:
    """Double the input"""
    return 2 * x

print(f">> Function's name: \"{compute.__name__}\"")
print(f">> Function's description: \"{compute.__doc__}\"")
print(">> Function's response with expected usage...")
print(compute(1.23456))
print(">> Function's response with expected error...")
print(compute())

>> Function's name: "compute"
>> Function's description: "Double the input"
>> Function's response with expected usage...
2.46912
>> Function's response with expected error...
Error executing "compute((); {})"
Exception: "TypeError("compute() missing 1 required positional argument: 'x'")"
Retries left: 1
Error executing "compute((); {})"
Exception: "TypeError("compute() missing 1 required positional argument: 'x'")"
Retries left: 0


Exception: Max attempts (2) reached without success

## Exercise 4 – REST Error-Handling Decorator

**Goal**: Implement a decorator `rest_error_handler` that wraps **REST-style
handler functions** and returns a standardized error response instead of letting
exceptions escape.

Assumptions:

- The wrapped function signature is `handler(message: str) -> dict`.
- On success, the function returns a dict like `{ "status": "ok", ... }`.
- If the function raises any exception, the decorator should:
  - Catch the exception.
  - Print/log an error message.
  - Return a dict like `{ "status": "error", "error": str(e) }`.

Requirements:

- `rest_error_handler` should be reusable on many different handlers.
- Don’t re-raise exceptions; convert them into error dicts.

Hints:

- This models what you did in `decorators.ipynb` for REST handlers.
- You can test with a handler that sometimes raises `ValueError` or `json.JSONDecodeError`.


In [14]:
# Exercise 4 – implement rest_error_handler here

from collections.abc import Callable
from typing import Any, Dict


def rest_error_handler(func: Callable[[str], Dict[str, Any]]) -> Callable[[str], Dict[str, Any]]:
    """Wrap a REST-style handler(message: str) -> dict with error handling.

    On exception, return {"status": "error", "error": str(e)} instead of raising.
    """

    def wrapper(message: str) -> Dict[str, Any]:
        # TODO: try calling func(message) and handle exceptions
        try: return func(message)
        except Exception as EXC:
            return dict(
                status = "error",
                error = repr(EXC)
            )

    return wrapper


# Example (uncomment after implementation):
import json

@rest_error_handler
def parse_json_message(msg: str) -> Dict[str, Any]:
    data = json.loads(msg)  # may raise
    return {"status": "ok", "length": len(data)}

print(parse_json_message("{\"a\": 1}"))
print(parse_json_message("not json"))

{'status': 'ok', 'length': 1}
{'status': 'error', 'error': "JSONDecodeError('Expecting value: line 1 column 1 (char 0)')"}


## Exercise 5 – Async WebSocket Handler Decorator

**Goal**: Implement an **async** decorator `async_ws_handler` that wraps WebSocket
message handlers.

Assumptions:

- Handler signature: `async def handler(message: str) -> None`.
- Decorator usage: `@async_ws_handler(stream_name="ticker-stream")` above the handler.
- Behavior:
  - When a message arrives, print `"[stream_name] WS message: {message}"`.
  - Call the original handler with `await handler(message)`.
  - If an exception is raised, catch it and print/log an error message, but **do not re-raise**.

Requirements:

- `async_ws_handler` should be a decorator factory that returns an **async wrapper**.
- You do **not** need to preserve metadata for this exercise (wrapping with `@wraps` is optional).

Hints:

- The wrapper must be `async def wrapper(message: str) -> None`.
- Inside the wrapper, use `await func(message)`.
- Test with a small `asyncio.run(main())` function that calls your wrapped handler.

In [22]:
# Exercise 5 – implement async_ws_handler here

from typing import Awaitable, Callable
from functools import wraps
import asyncio, datetime

WsHandler = Callable[[str], Awaitable[None]]


def async_ws_handler(stream_name: str) -> Callable[[WsHandler], WsHandler]:
    """Decorator factory for async WebSocket handlers.

    Logs incoming messages and catches exceptions.
    """

    def decorator(handler: WsHandler) -> WsHandler:
        @wraps(handler)
        async def wrapper(message: str) -> None:
            # TODO: print log message
            # TODO: await handler(message) and catch exceptions
            now_ts = datetime.datetime.now(datetime.timezone.utc)
            now = now_ts.isoformat(" ", "seconds")
            entry = f"[{now} | stream: \"{stream_name}\"]"
            print(f"{entry} WS message: \"{message}\"")
            try: await handler(message)
            except Exception as EXC:
                print(f"{entry} error: \"{repr(EXC)}\"")

        return wrapper  # type: ignore[return-value]

    return decorator


# Example (uncomment to test):
@async_ws_handler(stream_name="demo-stream")
async def ws_handler(msg: str) -> None:
    if "error" in msg:
        raise RuntimeError("simulated error")
    print("processing:", msg)

async def main():
    await ws_handler("hello")
    await ws_handler("this will error")

# In a notebook, use top-level await (Jupyter already runs an event loop).
# Use asyncio.run(main()) only when running this code as a script.
await main()

[2026-03-14 05:11:28+00:00 | stream: "demo-stream"] WS message: "hello"
processing: hello
[2026-03-14 05:11:28+00:00 | stream: "demo-stream"] WS message: "this will error"
[2026-03-14 05:11:28+00:00 | stream: "demo-stream"] error: "RuntimeError('simulated error')"


## Exercise 6 – Stacking Decorators and Call Order

**Goal**: Explore how the **order of stacked decorators** affects behavior.

You will:

1. Implement two simple decorators:
   - `decorator_a`: prints a message before and after calling the function.
   - `decorator_b`: prints a different message before and after.
2. Apply them in different orders:

   ```python
   @decorator_a
   @decorator_b
   def foo():
       ...

   @decorator_b
   @decorator_a
   def bar():
       ...
   ```

3. Observe the print order when calling `foo()` and `bar()`.

Requirements:

- In `decorator_a`, print e.g. `"A before"` and `"A after"`.
- In `decorator_b`, print e.g. `"B before"` and `"B after"`.
- Ensure both decorators return the inner result.

Hints:

- Remember: decorators stack **from bottom to top**:
  - `foo = decorator_a(decorator_b(foo))`.
- Use these prints to reason about call order and wrapping.

In [24]:
# Exercise 6 – implement decorator_a and decorator_b here

from collections.abc import Callable
from typing import Any


def decorator_a(func: Callable[..., Any]) -> Callable[..., Any]:
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        # TODO: print "A before"
        # TODO: call func
        # TODO: print "A after"
        # TODO: return result
        print("A before")
        result = func(*args, **kwargs)
        print("A after")
        return result

    return wrapper


def decorator_b(func: Callable[..., Any]) -> Callable[..., Any]:
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        # TODO: similar pattern, but with "B before" / "B after"
        print("B before")
        result = func(*args, **kwargs)
        print("B after")
        return result

    return wrapper


# After implementing, uncomment and run to see the order of prints:
@decorator_a
@decorator_b
def foo():
    print("foo body")
#
@decorator_b
@decorator_a
def bar():
    print("bar body")
#
print("Calling foo():")
foo()
print("\nCalling bar():")
bar()

Calling foo():
A before
B before
foo body
B after
A after

Calling bar():
B before
A before
bar body
A after
B after


## Exercise 7 – Simple Caching (Memoization) Decorator

**Goal**: Implement a decorator `memoize` that caches function results in memory.

Requirements:

- Use a dictionary `cache` inside the decorator to map from **arguments** to results.
- If a function is called again with the same arguments, return the cached result
  without recomputing.
- Support positional-only arguments for simplicity (ignore `**kwargs` for this exercise).
- Demonstrate the speedup on a slow function (e.g. one that calls `time.sleep`).

Hints:

- Use `args` (a tuple) as the cache key.
- Be careful not to put unhashable objects (like lists) into the cache key;
  for this exercise you can assume arguments are hashable.

This pattern is very useful for **CPU-bound** pure functions, such as pricing
formulas or expensive transforms that are called repeatedly with the same inputs.

In [25]:
# Exercise 7 – implement memoize here

from collections.abc import Callable
from typing import Any, Dict, Tuple


def memoize(func: Callable[..., Any]) -> Callable[..., Any]:
    """Simple memoization decorator for functions with hashable positional args.

    Cache is stored per-function in a closure.
    """
    cache: Dict[Tuple[Any, ...], Any] = {}

    def wrapper(*args: Any, **kwargs: Any) -> Any:
        # For this exercise, we ignore kwargs in the cache key
        key = args
        if key in cache: return cache[key]
        cache[key] = func(*args, **kwargs)
        # TODO: compute result, store in cache, and return it
        return cache[key]

    return wrapper


# Example (uncomment after implementation):
import time

@memoize
def slow_square(x: int) -> int:
    time.sleep(0.5)
    return x * x

t0 = time.perf_counter()
print(slow_square(10))  # first call: slow
print("elapsed:", time.perf_counter() - t0)

t0 = time.perf_counter()
print(slow_square(10))  # second call: should be fast (cached)
print("elapsed:", time.perf_counter() - t0)

100
elapsed: 0.5007038000039756
100
elapsed: 3.029999788850546e-05


## Exercise 8 – Caching Decorator (Class-Based)

**Goal**: Implement the same caching (memoization) behavior as in Exercise 7, but using a **decorating class** instead of a function. The cache should be an **attribute** of that class.

Requirements:

- Define a class (e.g. `Memoize`) that can be used as `@Memoize()` to decorate a function.
- Store the cache as an instance attribute (e.g. `self.cache`) so you can inspect or clear it from outside.
- If a function is called again with the same arguments, return the cached result without recomputing.
- Support positional-only arguments for simplicity (ignore `**kwargs` for this exercise).
- Demonstrate the speedup on a slow function (e.g. one that calls `time.sleep`).

Hints:

- The class must implement `__call__(self, func)` so that applying `@Memoize()` passes the wrapped function to the instance.
- In `__call__`, return a wrapper that uses `self.cache` as the cache dictionary.
- Use `args` (a tuple) as the cache key; assume arguments are hashable.

This exercise shows how a **callable class** can act as a decorator and keep state (the cache) in a natural way.

In [27]:
# Exercise 8 – implement Memoize class here

from collections.abc import Callable
from typing import Any, Dict, Tuple


class Memoize:
    """Class-based memoization decorator. Cache is stored in self.cache."""

    def __init__(self) -> None:
        # TODO: initialize the cache as an instance attribute (e.g. self.cache = {})
        self.cache: Dict[Tuple[Any, ...], Any] = {}

    def __call__(self, func: Callable[..., Any]) -> Callable[..., Any]:
        # TODO: return a wrapper that uses self.cache to memoize func(*args)
        #       (same logic as Exercise 7, but read/write self.cache)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            key = args
            if key in self.cache:
                return self.cache[key]
            self.cache[key] = func(*args, **kwargs)
            return self.cache[key]
        return wrapper


# Example (uncomment after implementation):
import time

# TODO: Initialize an instance of "Memoize" to use as decorator.
memo = Memoize()

@memo
def slow_square(x: int) -> int:
    time.sleep(0.5)
    return x * x

t0 = time.perf_counter()
print(slow_square(10))  # first call: slow
print("elapsed:", time.perf_counter() - t0)

t0 = time.perf_counter()
print(slow_square(10))  # second call: fast (cached)
print("elapsed:", time.perf_counter() - t0)

# Cache is an attribute of the decorator instance:
print("cache:", memo.cache)

100
elapsed: 0.5008815000182949
100
elapsed: 4.439998883754015e-05
cache: {(10,): 100}
